In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
import lightgbm as lgb
from catboost import CatBoostRegressor

# Chargement (adapte les chemins si besoin)
X_train_raw = pd.read_csv("../features/global_features/X_train.csv")
y_train_raw = pd.read_csv("../features/global_features/y_train.csv")
X_test_raw = pd.read_csv("../features/global_features/X_test.csv")

# Nettoyage des IDs et index
X = X_train_raw.drop(columns=['video_id', 'Unnamed: 0'], errors='ignore')
X_test_final = X_test_raw.drop(columns=['video_id', 'Unnamed: 0'], errors='ignore')
y = y_train_raw.values.flatten()

# Imputation (fondamental pour la stabilité)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_test_imputed = imputer.transform(X_test_final)

# Conversion en DataFrame pour garder les noms de colonnes
X = pd.DataFrame(X_imputed, columns=X.columns)
X_test_final = pd.DataFrame(X_test_imputed, columns=X_test_final.columns)

In [2]:
def objective_lgb(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "num_leaves": trial.suggest_int("num_leaves", 7, 31), # Petit pour éviter l'overfitting
        "max_depth": trial.suggest_int("max_depth", 3, 6),   # Arbres très courts
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 0.4), # SEULEMENT 10-40% des features
        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X):
        model = lgb.LGBMRegressor(n_estimators=1000, **params)
        model.fit(X.iloc[tr_idx], y[tr_idx], 
                  eval_set=[(X.iloc[val_idx], y[val_idx])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        preds = model.predict(X.iloc[val_idx])
        scores.append(np.sqrt(mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(objective_lgb, n_trials=50)

[I 2026-03-03 23:02:50,446] A new study created in memory with name: no-name-74f0d1ed-a087-49d9-a00f-8eb623268de6
[I 2026-03-03 23:03:09,319] Trial 0 finished with value: 1.2555204961180197 and parameters: {'learning_rate': 0.0425425379559799, 'num_leaves': 26, 'max_depth': 6, 'min_child_samples': 26, 'feature_fraction': 0.26124056808454055, 'reg_alpha': 2.9570422623786983, 'reg_lambda': 2.6376113884874663}. Best is trial 0 with value: 1.2555204961180197.
[I 2026-03-03 23:03:20,024] Trial 1 finished with value: 1.2457178696611992 and parameters: {'learning_rate': 0.0430788948880814, 'num_leaves': 7, 'max_depth': 4, 'min_child_samples': 24, 'feature_fraction': 0.328084694149419, 'reg_alpha': 2.1335215267213754, 'reg_lambda': 0.24871296410290758}. Best is trial 1 with value: 1.2457178696611992.
[I 2026-03-03 23:03:34,304] Trial 2 finished with value: 1.247443606362084 and parameters: {'learning_rate': 0.04243282300832153, 'num_leaves': 31, 'max_depth': 5, 'min_child_samples': 45, 'featur

In [3]:
# Entraînement final avec les meilleurs paramètres
best_lgb = lgb.LGBMRegressor(n_estimators=2000, **study_lgb.best_params)
# On fit sur TOUT le train
best_lgb.fit(X, y)


# Moyenne pondérée (Blending)
preds_lgb = best_lgb.predict(X_test_final)

In [5]:
# Affiche le meilleur score RMSE moyen obtenu pendant l'optimisation
print(f"RMSE estimé (CV) : {study_lgb.best_value:.4f}")

RMSE estimé (CV) : 1.2335
